# YOLOv8m — FieldPlant Object Detection

Training on the **FieldPlant** dataset (manhhoangvan/fieldplant on Kaggle).
Detects 27 crop disease classes across Cassava, Corn, and Tomato.

**Setup:**
1. Add dataset: `manhhoangvan/fieldplant`
2. Enable **GPU T4 x2** (or P100)
3. Run All

In [ ]:
# Cell 1 — GPU + environment check
import os

print('=== GPU INFO ===')
!nvidia-smi

print('\n=== KAGGLE INPUT ===')
!ls /kaggle/input/

print('\n=== DATASET STRUCTURE ===')
base = '/kaggle/input/fieldplant'
if os.path.exists(base):
    for root, dirs, files in os.walk(base):
        depth = root.replace(base, '').count(os.sep)
        if depth > 2:
            continue
        indent = '  ' * depth
        print(f'{indent}{os.path.basename(root)}/')
        if depth == 2:
            print(f'{indent}  ... ({len(files)} files)')
else:
    print('Dataset not found at /kaggle/input/fieldplant — check ls output above.')

print('\n=== EXISTING data.yaml ===')
!cat /kaggle/input/fieldplant/data.yaml 2>/dev/null || echo 'No data.yaml found'

In [ ]:
# Cell 2 — Install ultralytics
!pip install -q ultralytics==8.3.0

import ultralytics
ultralytics.checks()

In [ ]:
# Cell 3 — Configuration
import torch
import yaml
import shutil
import random
from pathlib import Path

# ═══════════════════════════════════════════════════════
#  CONFIGURATION
# ═══════════════════════════════════════════════════════
DATASET_ROOT = Path('/kaggle/input/fieldplant')
WORK_DIR     = Path('/kaggle/working')

MODEL        = 'yolov8m.pt'
IMGSZ        = 640
EPOCHS       = 50
BATCH        = 16             # reduce to 8 if OOM
WORKERS      = 4
PATIENCE     = 15
LR0          = 1e-2
LRF          = 1e-2
OPTIMIZER    = 'AdamW'
VAL_SPLIT    = 0.15           # fraction of train used as val when no val/ folder exists
SEED         = 42
DEVICE       = '0,1' if torch.cuda.device_count() > 1 else '0'
PROJECT_NAME = 'yolov8m_fieldplant'

CLASS_NAMES = [
    'Cassava Bacterial Blight',
    'Cassava Brown Leaf Spot',
    'Cassava Healthy',
    'Cassava Mosaic',
    'Cassava Root Rot',
    'Corn Brown Spots',
    'Corn Charcoal',
    'Corn Chlorotic Leaf Spot',
    'Corn Gray leaf spot',
    'Corn Healthy',
    'Corn Insects Damages',
    'Corn Mildew',
    'Corn Purple Discoloration',
    'Corn Smut',
    'Corn Streak',
    'Corn Stripe',
    'Corn Violet Decoloration',
    'Corn Yellow Spots',
    'Corn Yellowing',
    'Corn leaf blight',
    'Corn rust leaf',
    'Tomato Brown Spots',
    'Tomato bacterial wilt',
    'Tomato blight leaf',
    'Tomato healthy',
    'Tomato leaf mosaic virus',
    'Tomato leaf yellow virus',
]

assert len(CLASS_NAMES) == 27
print(f'Torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()} — {torch.cuda.device_count()} GPU(s)')
print(f'Device : {DEVICE}')
print(f'Classes: {len(CLASS_NAMES)}')

In [ ]:
# Cell 4 — Detect dataset layout and build train/val split if needed
#
# The FieldPlant Kaggle dataset ships with only a train/ folder.
# We split it 85/15 into train and val by symlinking label+image pairs.

train_img_src = DATASET_ROOT / 'train' / 'images'
train_lbl_src = DATASET_ROOT / 'train' / 'labels'

val_img_src   = DATASET_ROOT / 'valid' / 'images'
val_lbl_src   = DATASET_ROOT / 'valid' / 'labels'

HAS_VAL  = val_img_src.exists() and any(val_img_src.iterdir())
HAS_TEST = (DATASET_ROOT / 'test' / 'images').exists()

print(f'train/ exists : {train_img_src.exists()}')
print(f'valid/ exists : {HAS_VAL}')
print(f'test/  exists : {HAS_TEST}')

if not HAS_VAL:
    print(f'\nNo valid/ split found — creating {int(VAL_SPLIT*100)}% val split from train...')

    # Collect all images that have a corresponding label
    all_imgs = sorted([
        p for p in train_img_src.glob('*.[jp][pn][g]')
        if (train_lbl_src / (p.stem + '.txt')).exists()
    ])

    random.seed(SEED)
    random.shuffle(all_imgs)
    n_val   = int(len(all_imgs) * VAL_SPLIT)
    val_set = set(p.stem for p in all_imgs[:n_val])

    # Create writable split dirs under /kaggle/working
    for split in ['train_split', 'val_split']:
        for sub in ['images', 'labels']:
            (WORK_DIR / split / sub).mkdir(parents=True, exist_ok=True)

    for img_path in all_imgs:
        lbl_path = train_lbl_src / (img_path.stem + '.txt')
        split    = 'val_split' if img_path.stem in val_set else 'train_split'
        shutil.copy2(img_path,  WORK_DIR / split / 'images' / img_path.name)
        shutil.copy2(lbl_path,  WORK_DIR / split / 'labels' / lbl_path.name)

    TRAIN_PATH = str(WORK_DIR / 'train_split' / 'images')
    VAL_PATH   = str(WORK_DIR / 'val_split'   / 'images')
    n_train = len(all_imgs) - n_val
    print(f'  Train: {n_train} images')
    print(f'  Val  : {n_val} images')
else:
    TRAIN_PATH = str(train_img_src)
    VAL_PATH   = str(val_img_src)
    print(f'Using existing splits.')
    print(f'  Train: {len(list(train_img_src.glob("*.[jp][pn][g]")))}')
    print(f'  Val  : {len(list(val_img_src.glob("*.[jp][pn][g]")))}')

TEST_PATH = str(DATASET_ROOT / 'test' / 'images') if HAS_TEST else VAL_PATH
print(f'  Test : {"separate test/" if HAS_TEST else "using val set"}')

In [ ]:
# Cell 5 — Write data.yaml
data_yaml = {
    'train': TRAIN_PATH,
    'val'  : VAL_PATH,
    'test' : TEST_PATH,
    'nc'   : 27,
    'names': CLASS_NAMES,
}

yaml_path = WORK_DIR / 'fieldplant.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f'data.yaml written → {yaml_path}')
!cat /kaggle/working/fieldplant.yaml

In [ ]:
# Cell 6 — Class distribution (train set)
import collections
import matplotlib.pyplot as plt

# Use whichever label dir we're actually training on
if not HAS_VAL:
    lbl_dir = WORK_DIR / 'train_split' / 'labels'
else:
    lbl_dir = train_lbl_src

counts = collections.Counter()
for lf in lbl_dir.glob('*.txt'):
    with open(lf) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                counts[int(parts[0])] += 1

print(f'Total bounding boxes in train: {sum(counts.values()):,}\n')
for idx in sorted(counts):
    bar = '█' * (counts[idx] // max(1, max(counts.values()) // 30))
    print(f'  [{idx:2d}] {CLASS_NAMES[idx]:<35s} {counts[idx]:5d}  {bar}')

fig, ax = plt.subplots(figsize=(14, 5))
xs = [CLASS_NAMES[i] for i in sorted(counts)]
ys = [counts[i] for i in sorted(counts)]
ax.bar(xs, ys, color='steelblue', edgecolor='white', linewidth=0.4)
ax.set_title('Train set — Bounding boxes per class', fontsize=13)
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(WORK_DIR / 'class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Cell 7 — Train YOLOv8m
from ultralytics import YOLO

model = YOLO(MODEL)

results = model.train(
    data         = str(yaml_path),
    epochs       = EPOCHS,
    imgsz        = IMGSZ,
    batch        = BATCH,
    workers      = WORKERS,
    patience     = PATIENCE,
    optimizer    = OPTIMIZER,
    lr0          = LR0,
    lrf          = LRF,
    device       = DEVICE,
    project      = str(WORK_DIR / 'runs'),
    name         = PROJECT_NAME,
    exist_ok     = True,
    save         = True,
    save_period  = 10,
    plots        = True,
    val          = True,
    # Augmentation
    hsv_h        = 0.015,
    hsv_s        = 0.7,
    hsv_v        = 0.4,
    degrees      = 5.0,
    translate    = 0.1,
    scale        = 0.5,
    fliplr       = 0.5,
    flipud       = 0.1,
    mosaic       = 1.0,
    mixup        = 0.1,
    close_mosaic = 10,
    verbose      = True,
)

best_weights = WORK_DIR / 'runs' / PROJECT_NAME / 'weights' / 'best.pt'
print(f'\n✅ Training complete!')
print(f'Best weights → {best_weights}')

In [ ]:
# Cell 8 — Val evaluation with best.pt
from ultralytics import YOLO

best_weights = WORK_DIR / 'runs' / PROJECT_NAME / 'weights' / 'best.pt'
model_best   = YOLO(str(best_weights))

val_metrics = model_best.val(
    data      = str(yaml_path),
    split     = 'val',
    imgsz     = IMGSZ,
    batch     = BATCH,
    device    = DEVICE,
    conf      = 0.001,
    iou       = 0.6,
    plots     = True,
    save_json = True,
)

print('\n=== VALIDATION METRICS ===')
print(f'mAP@50     : {val_metrics.box.map50:.4f}')
print(f'mAP@50-95  : {val_metrics.box.map:.4f}')
print(f'Precision  : {val_metrics.box.mp:.4f}')
print(f'Recall     : {val_metrics.box.mr:.4f}')

print('\n--- Per-class AP@50 ---')
for name, ap in zip(CLASS_NAMES, val_metrics.box.ap50):
    print(f'  {name:<35s}: {ap:.4f}')

In [ ]:
# Cell 9 — Visualise training curves + confusion matrix
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

run_dir = WORK_DIR / 'runs' / PROJECT_NAME

plot_files = [
    ('results.png',                   'Training Curves'),
    ('confusion_matrix.png',          'Confusion Matrix'),
    ('confusion_matrix_normalized.png','Confusion Matrix (Normalised)'),
    ('PR_curve.png',                  'Precision-Recall Curve'),
    ('F1_curve.png',                  'F1 Curve'),
]

for fname, title in plot_files:
    fp = run_dir / fname
    if not fp.exists():
        print(f'  (skipping {fname} — not found)')
        continue
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.imshow(mpimg.imread(fp))
    ax.axis('off')
    ax.set_title(title, fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 10 — Sample inference on val images
import random, matplotlib.pyplot as plt
from pathlib import Path

sample_dir  = Path(VAL_PATH)
all_imgs    = sorted(sample_dir.glob('*.[jp][pn][g]'))
sample_imgs = random.sample(all_imgs, min(8, len(all_imgs)))

preds = model_best.predict(
    source  = [str(p) for p in sample_imgs],
    imgsz   = IMGSZ,
    conf    = 0.25,
    iou     = 0.45,
    device  = DEVICE,
    save    = False,
    verbose = False,
)

n    = len(preds)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4))
axes = axes.flatten()

for ax, r in zip(axes, preds):
    ax.imshow(r.plot()[:, :, ::-1])
    ax.axis('off')
    ax.set_title(f'{Path(r.path).name}\n{len(r.boxes)} det', fontsize=8)

for ax in axes[n:]:
    ax.set_visible(False)

plt.suptitle('Sample Detections (conf ≥ 0.25)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(WORK_DIR / 'sample_detections.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 11 — Results summary
print('=' * 55)
print('  FINAL RESULTS SUMMARY')
print('=' * 55)
print(f'  Model      : YOLOv8m')
print(f'  Dataset    : FieldPlant (manhhoangvan/fieldplant)')
print(f'  Classes    : {len(CLASS_NAMES)}')
print(f'  Image size : {IMGSZ}x{IMGSZ}')
print(f'  Val split  : {"existing valid/" if HAS_VAL else f"{int(VAL_SPLIT*100)}% of train"}')
print()
print(f'  mAP@50     : {val_metrics.box.map50:.4f}')
print(f'  mAP@50-95  : {val_metrics.box.map:.4f}')
print(f'  Precision  : {val_metrics.box.mp:.4f}')
print(f'  Recall     : {val_metrics.box.mr:.4f}')
print('=' * 55)
print()
print('  WHAT TO COPY INTO src/config.py')
print()
print('  YOLO_PATH: str = "models/best_yolo8m.pt"')
print('  YOLO_FALLBACK_MODEL: str = "yolov8m.pt"')
print('=' * 55)

In [ ]:
# Cell 12 — Package outputs into a zip for download
import shutil, zipfile

output_dir = WORK_DIR / 'outputs'
output_dir.mkdir(exist_ok=True)

run_dir = WORK_DIR / 'runs' / PROJECT_NAME

# Weights
weights_out = output_dir / 'weights'
weights_out.mkdir(exist_ok=True)
for wf in (run_dir / 'weights').glob('*.pt'):
    shutil.copy(wf, weights_out / wf.name)
    print(f'Copied weight : {wf.name}')

# Plots
plots_out = output_dir / 'plots'
plots_out.mkdir(exist_ok=True)
for pf in [
    *run_dir.glob('*.png'),
    WORK_DIR / 'class_distribution.png',
    WORK_DIR / 'sample_detections.png',
]:
    pf = Path(pf)
    if pf.exists():
        shutil.copy(pf, plots_out / pf.name)
        print(f'Copied plot   : {pf.name}')

# CSVs and JSON
for rf in list(run_dir.glob('*.csv')) + list(run_dir.glob('*.json')):
    shutil.copy(rf, output_dir / rf.name)
    print(f'Copied result : {rf.name}')

# Zip
zip_path = WORK_DIR / f'{PROJECT_NAME}_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in output_dir.rglob('*'):
        if f.is_file():
            zf.write(f, f.relative_to(output_dir))

zip_mb = zip_path.stat().st_size / (1024 * 1024)
print(f'\n✅ ZIP ready : {zip_path}  ({zip_mb:.1f} MB)')
print()
print('Download from the Output tab (right side panel) →')
print('  weights/best.pt  →  models/best_yolo8m.pt  in your project')
print('  plots/           →  check training curves & confusion matrix')